<a href="https://colab.research.google.com/github/avyue/datasci112_finalproject/blob/main/LA_CoC_Shelter_Count_%26_Housing_Inventory_Count_(vacant_occupied%2C_utilization_rate%2C_PIT_count).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#all info stored in LA CoC + derived occupied vs unoccpied beds from PIT count and utilization rate

import io, requests, numpy as np, pandas as pd
pd.set_option("display.max_columns", 200)

HIC_URL = "https://www.lahsa.org/item.ashx?id=9369-housing-inventory-count-hic-.xlsx&dl=true"
content = requests.get(HIC_URL, headers={"User-Agent": "Mozilla/5.0"}).content
xls = pd.ExcelFile(io.BytesIO(content))
df = pd.read_excel(xls, sheet_name="2025 HIC - All Projects", header=0)
df.columns = df.columns.astype(str).str.strip()
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
df = df.dropna(subset=["Proj. Type", "Total Beds"]).copy()
print(df.shape, "= every original column kept")

(1185, 78) = every original column kept


In [8]:
# only beds that actually exist now (drop "Under Development")
df = df[df["Inventory Type"].astype(str).str.strip().str.lower().eq("current")].copy()

for c in ["Total Beds", "PIT Count", "Utilization Rate"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df.shape)

(1176, 78)


In [9]:
# occupied = people present on count night (PIT Count); vacant = capacity not in use
df["occupied_beds"] = df["PIT Count"]
df["vacant_beds"]   = (df["Total Beds"] - df["PIT Count"]).clip(lower=0)   # clip: a few projects report over capacity

# cross-check against the published Utilization Rate (occupied/total should ≈ Utilization Rate)
df["occupancy_rate_check"] = (df["occupied_beds"] / df["Total Beds"]).round(3)

# helpful extras for grouping later
df["bed_category"] = np.where(df["Proj. Type"].astype(str).str.strip().isin(["ES","TH","SH"]),
                              "Shelter", "Permanent Housing")
df["spa_num"] = df["SPA"].astype(str).str.extract(r"(\d+)")[0]

print(df[["Proj. Type","Total Beds","PIT Count","occupied_beds","vacant_beds",
          "Utilization Rate","occupancy_rate_check","bed_category","spa_num"]].head())

  Proj. Type  Total Beds  PIT Count  occupied_beds  vacant_beds  \
0         ES        33.0       33.0           33.0          0.0   
1         ES        15.0       15.0           15.0          0.0   
2         ES        20.0       20.0           20.0          0.0   
3         ES         6.0        3.0            3.0          3.0   
4        RRH         9.0        9.0            9.0          0.0   

   Utilization Rate  occupancy_rate_check       bed_category spa_num  
0               1.0                   1.0            Shelter       3  
1               1.0                   1.0            Shelter       4  
2               1.0                   1.0            Shelter       8  
3               0.5                   0.5            Shelter       5  
4               1.0                   1.0  Permanent Housing       8  


In [10]:
df.to_csv("hic_2025_FULL_project_level.csv", index=False)
print("saved:", df.shape, "columns:", len(df.columns))

from google.colab import files
files.download("hic_2025_vacant/occupied_beds.csv")

saved: (1176, 83) columns: 83


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>